# PicoRV32 + CIM SoC — 宿主机环境准备

本 notebook 在**宿主机** (有 RISC-V GCC + Python) 上运行，完成：

1. 训练 Small MLP (784→16→10) 并量化导出
2. Python golden model 验证 20 张测试图
3. **批量编译 20 个 firmware.hex** (每张图一个)
4. 打包文件，准备复制到 VCS 虚拟机 和 PYNQ 板

环境要求：
- RISC-V GCC (riscv64-elf-gcc 或 riscv64-unknown-elf-gcc)
- Python 3 + PyTorch + torchvision

VCS 虚拟机**不需要**任何编译工具，只需要 VCS。

## 0. 路径设置

In [1]:
import os, sys, subprocess, shutil, struct
import numpy as np

# 在 picorv32/fw/ 目录下运行
FW_DIR = os.path.abspath(".")
PROJ_ROOT = os.path.dirname(FW_DIR)  # picorv32/
DATA_DIR = "../picorv32/fw/small_mlp_data"
N_IMAGES = 20
OUT_DIR = "fw_hex_batch"
FW_DIR = os.path.abspath("../picorv32/fw")
fw_hex = os.path.join(FW_DIR, "firmware.hex")
#assert os.path.exists(os.path.join(FW_DIR, "../picorv32/fw/firmware.c"))

## 1. 检查工具链

In [2]:
# RISC-V GCC
rv_gcc = None
cross_prefix = None
for prefix in ["riscv64-elf-", "riscv64-unknown-elf-", "riscv32-unknown-elf-"]:
    if shutil.which(f"{prefix}gcc"):
        rv_gcc = f"{prefix}gcc"
        cross_prefix = prefix
        break

if rv_gcc:
    ver = subprocess.check_output([rv_gcc, "--version"], text=True).split("\n")[0]
    print(f"✓ RISC-V GCC: {ver}")
    print(f"  CROSS prefix: {cross_prefix}")
else:
    print("✗ RISC-V GCC 未找到!")
    print("  Arch:   sudo pacman -S riscv64-elf-gcc")
    print("  Debian: sudo apt install gcc-riscv64-unknown-elf")

# Python
try:
    import torch; print(f"✓ PyTorch {torch.__version__}")
except ImportError:
    print("✗ PyTorch: pip install torch torchvision")

try:
    from torchvision import datasets; print("✓ torchvision")
except ImportError:
    print("✗ torchvision: pip install torchvision")

✓ RISC-V GCC: riscv64-elf-gcc (Arch Linux Repositories) 15.2.0
  CROSS prefix: riscv64-elf-
✓ PyTorch 2.10.0+cu128
✓ torchvision


## 2. 训练 Small MLP 并量化

In [3]:
if not os.path.isdir(DATA_DIR):
    print(f"训练模型并导出到 {DATA_DIR}/ ...")
    ret = subprocess.run(
        ["python3", "../picorv32/fw/small_mlp_quantize.py",
         "--output-dir", DATA_DIR,
         "--num-test", str(N_IMAGES),
         "--seed", "42"],
        text=True
    )
    assert ret.returncode == 0, "训练失败!"
else:
    print(f"{DATA_DIR}/ 已存在, 跳过训练")

# 列出文件
for f in sorted(os.listdir(DATA_DIR)):
    full = os.path.join(DATA_DIR, f)
    if os.path.isfile(full):
        print(f"  {f:30s} {os.path.getsize(full):>8d} B")
    else:
        n = len(os.listdir(full))
        print(f"  {f + '/':30s} ({n} files)")

训练模型并导出到 ../picorv32/fw/small_mlp_data/ ...


100.0%
100.0%
100.0%
100.0%


Training SmallMLP (784→16→10)
  Epoch 1/20: loss=0.6961, acc=91.16%
  Epoch 2/20: loss=0.3143, acc=91.99%
  Epoch 3/20: loss=0.2738, acc=92.58%
  Epoch 4/20: loss=0.2474, acc=93.16%
  Epoch 5/20: loss=0.2299, acc=93.47%
  Epoch 6/20: loss=0.2166, acc=93.73%
  Epoch 7/20: loss=0.2055, acc=94.07%
  Epoch 8/20: loss=0.1963, acc=94.21%
  Epoch 9/20: loss=0.1880, acc=94.43%
  Epoch 10/20: loss=0.1819, acc=94.44%
  Epoch 11/20: loss=0.1753, acc=94.54%
  Epoch 12/20: loss=0.1708, acc=94.72%
  Epoch 13/20: loss=0.1651, acc=94.75%
  Epoch 14/20: loss=0.1605, acc=94.77%
  Epoch 15/20: loss=0.1571, acc=94.91%
  Epoch 16/20: loss=0.1531, acc=94.81%
  Epoch 17/20: loss=0.1503, acc=94.93%
  Epoch 18/20: loss=0.1473, acc=95.02%
  Epoch 19/20: loss=0.1440, acc=95.29%
  Epoch 20/20: loss=0.1423, acc=95.15%

Quantizing...
  FC1: s_w=0.010215, s_out=0.268343, mult=10, shift=16
  FC2: s_w=0.012717, s_out=0.332514, mult=673, shift=16

INT8 accuracy:
  Float: 95.15%, INT8: 95.11%
  [0000] label=7, pred=7 ✓


## 3. Golden Model 验证 (20 张图)

In [4]:
def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

# 量化参数
qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

# 解包权重
def unpack_weights(chunk_path, out_dim, in_dim):
    chunks = read_hex_u32(chunk_path)
    w = np.zeros((out_dim, in_dim), dtype=np.int8)
    idx = 0
    for ob in range((out_dim + 15) // 16):
        for ib in range((in_dim + 15) // 16):
            for ch in range(64):
                r, cg = ch // 4, ch % 4
                word = chunks[idx]; idx += 1
                for b in range(4):
                    oi = ob * 16 + r
                    ii = ib * 16 + cg * 4 + b
                    if oi < out_dim and ii < in_dim:
                        val = (word >> (b*8)) & 0xFF
                        w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
    return w

def unpack_bias(path):
    return np.array([np.int32(np.uint32(v)) for v in read_hex_u32(path)], dtype=np.int32)

def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
    x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
    acc = w_i8.astype(np.int32) @ x_eff + b_i32
    if relu: acc = np.maximum(acc, 0)
    out = np.zeros(len(acc), dtype=np.int8)
    for i in range(len(acc)):
        prod = int(acc[i]) * int(mult)
        shifted = (prod + (1 << (shift-1))) >> shift if shift > 0 else prod
        out[i] = np.int8(max(-128, min(127, shifted)))
    return out

w1 = unpack_weights(f"{DATA_DIR}/fc1_weight_tiles.hex", 16, 784)
b1 = unpack_bias(f"{DATA_DIR}/fc1_bias.hex")
w2 = unpack_weights(f"{DATA_DIR}/fc2_weight_tiles.hex", 10, 16)
b2 = unpack_bias(f"{DATA_DIR}/fc2_bias.hex")

print(f"FC1: w={w1.shape}, b={b1.shape}, mult={fc1_mult}, shift={fc1_shift}")
print(f"FC2: w={w2.shape}, b={b2.shape}, mult={fc2_mult}, shift={fc2_shift}")
print()

golden_results = []
for i in range(N_IMAGES):
    prefix = f"img_{i:04d}"
    img = np.array(read_hex_u8(f"{DATA_DIR}/test_images/{prefix}.hex"), dtype=np.uint8)
    label = int(open(f"{DATA_DIR}/test_images/{prefix}_label.txt").read().strip())
    fc1 = hw_mvm(img, w1, b1, hw_zp1, fc1_mult, fc1_shift, True)
    fc2 = hw_mvm(fc1.view(np.uint8), w2, b2, hw_zp2, fc2_mult, fc2_shift, False)
    pred = int(np.argmax(fc2))
    golden_results.append((i, label, pred, pred == label))
    print(f"  [{i:04d}] label={label}, pred={pred} {'✓' if pred==label else '✗'}")

correct = sum(1 for r in golden_results if r[3])
print(f"\nGolden model 准确率: {correct}/{N_IMAGES} ({100*correct/N_IMAGES:.0f}%)")

FC1: w=(16, 784), b=(16,), mult=10, shift=16
FC2: w=(10, 16), b=(10,), mult=673, shift=16

  [0000] label=7, pred=7 ✓
  [0001] label=2, pred=2 ✓
  [0002] label=1, pred=1 ✓
  [0003] label=0, pred=0 ✓
  [0004] label=4, pred=4 ✓
  [0005] label=1, pred=1 ✓
  [0006] label=4, pred=4 ✓
  [0007] label=9, pred=9 ✓
  [0008] label=5, pred=6 ✗
  [0009] label=9, pred=9 ✓
  [0010] label=0, pred=0 ✓
  [0011] label=6, pred=6 ✓
  [0012] label=9, pred=9 ✓
  [0013] label=0, pred=0 ✓
  [0014] label=1, pred=1 ✓
  [0015] label=5, pred=5 ✓
  [0016] label=9, pred=9 ✓
  [0017] label=7, pred=7 ✓
  [0018] label=3, pred=8 ✗
  [0019] label=4, pred=4 ✓

Golden model 准确率: 18/20 (90%)


## 4. 批量编译 20 个 firmware.hex

In [5]:
os.makedirs(OUT_DIR, exist_ok=True)
for i in range(N_IMAGES):
    idx = f"{i:04d}"
    label = golden_results[i][1]
    subprocess.run(["make", "-s", "clean"], check=False,
                   capture_output=True, cwd=FW_DIR)
    ret = subprocess.run(
        ["make", "-s", "DATA_DIR=small_mlp_data", f"IMAGE_IDX={i}",
         f"CROSS={cross_prefix}"],
        capture_output=True, text=True, cwd=FW_DIR
    )
    if ret.returncode != 0 or not os.path.exists(fw_hex):
        print(f"  [{idx}] 编译失败!")
        print(ret.stderr)
        continue
    shutil.copy(fw_hex, f"{OUT_DIR}/firmware_{idx}.hex")
    n_words = sum(1 for l in open(fw_hex) if l.strip())
    print(f"  [{idx}] label={label}, {n_words} words ({n_words*4} bytes)")

  [0000] label=7, 3773 words (15092 bytes)
  [0001] label=2, 3773 words (15092 bytes)
  [0002] label=1, 3773 words (15092 bytes)
  [0003] label=0, 3773 words (15092 bytes)
  [0004] label=4, 3773 words (15092 bytes)
  [0005] label=1, 3773 words (15092 bytes)
  [0006] label=4, 3773 words (15092 bytes)
  [0007] label=9, 3773 words (15092 bytes)
  [0008] label=5, 3773 words (15092 bytes)
  [0009] label=9, 3773 words (15092 bytes)
  [0010] label=0, 3773 words (15092 bytes)
  [0011] label=6, 3773 words (15092 bytes)
  [0012] label=9, 3773 words (15092 bytes)
  [0013] label=0, 3773 words (15092 bytes)
  [0014] label=1, 3773 words (15092 bytes)
  [0015] label=5, 3773 words (15092 bytes)
  [0016] label=9, 3773 words (15092 bytes)
  [0017] label=7, 3773 words (15092 bytes)
  [0018] label=3, 3773 words (15092 bytes)
  [0019] label=4, 3773 words (15092 bytes)


## 4.5 保存 Golden Results

将 golden model 的推理结果写入 `fw_hex_batch/golden_results.txt`，
供 PYNQ 端 notebook 和 VCS 仿真对比使用。

In [6]:
# ---- Save golden results to fw_hex_batch/ ----
golden_file = os.path.join(OUT_DIR, "golden_results.txt")

with open(golden_file, "w") as f:
    f.write("# Golden Model Results (bit-accurate INT8, small_mlp 784->16->10)\n")
    f.write(f"# Generated by prepare_picorv32_env.ipynb\n")
    f.write(f"# Quant: fc1=({fc1_mult},{fc1_shift}), fc2=({fc2_mult},{fc2_shift})\n")
    f.write(f"# ZP: fc1={hw_zp1}, fc2={hw_zp2}\n")
    f.write(f"# Format: idx label pred correct\n")
    for i, label, pred, ok in golden_results:
        f.write(f"{i} {label} {pred} {1 if ok else 0}\n")

correct = sum(1 for r in golden_results if r[3])
print(f"Saved: {golden_file}")
print(f"  {N_IMAGES} images, accuracy {correct}/{N_IMAGES} ({100*correct/N_IMAGES:.0f}%)")
print()
print("内容预览:")
with open(golden_file) as f:
    print(f.read())

Saved: fw_hex_batch/golden_results.txt
  20 images, accuracy 18/20 (90%)

内容预览:
# Golden Model Results (bit-accurate INT8, small_mlp 784->16->10)
# Generated by prepare_picorv32_env.ipynb
# Quant: fc1=(10,16), fc2=(673,16)
# ZP: fc1=0, fc2=0
# Format: idx label pred correct
0 7 7 1
1 2 2 1
2 1 1 1
3 0 0 1
4 4 4 1
5 1 1 1
6 4 4 1
7 9 9 1
8 5 6 0
9 9 9 1
10 0 0 1
11 6 6 1
12 9 9 1
13 0 0 1
14 1 1 1
15 5 5 1
16 9 9 1
17 7 7 1
18 3 8 0
19 4 4 1



## 5. 检查 BRAM 占用

In [7]:
# 用 image 0 检查
n_words = sum(1 for l in open(f"{OUT_DIR}/firmware_0000.hex") if l.strip())
n_bytes = n_words * 4
print(f"firmware.hex: {n_words} words = {n_bytes} bytes ({n_bytes/1024:.1f} KB)")
print(f"BRAM 容量:    8192 words = 32768 bytes (32.0 KB)")
print(f"占用率:       {100*n_bytes/32768:.1f}%")
print()
if n_bytes > 32768:
    print("⚠ 超出 32KB BRAM!")
else:
    print("✓ 固件可以放入 BRAM")

firmware.hex: 3773 words = 15092 bytes (14.7 KB)
BRAM 容量:    8192 words = 32768 bytes (32.0 KB)
占用率:       46.1%

✓ 固件可以放入 BRAM


## 6. 下一步

文件准备好了，需要复制到两个地方:

### → VCS 虚拟机 (仿真验证)
```
复制到虚拟机:
  picorv32/fw/fw_hex_batch/        ← 20 个 firmware hex
  picorv32/hw/                     ← RTL + tb + scripts
  hw/rtl/                          ← CIM IP

在虚拟机上运行:
  cd picorv32/
  bash hw/scripts/run_tb_rv32_batch.sh
```

### → PYNQ 板 (上板交叉验证)
```
上传到 PYNQ Jupyter:
  cim_rv32_soc.bit       ← bitstream
  small_mlp_data/                  ← 模型数据
  fw_hex_batch/golden_results.txt  ← golden 结果
  cim_driver.py                    ← 驱动
  picorv32_small_mlp_pynq.ipynb    ← 测试 notebook
```